In [ ]:
#pip install openmeteo-requests retry-requests requests-cache pandas

Coordinates:
* Mawsynram, India: (25.2985, 91.5823) https://open-meteo.com/en/docs/historical-weather-api?timezone=auto&latitude=25.2985&longitude=91.5823&hourly=temperature_2m,relative_humidity_2m,rain,wind_speed_10m,cloud_cover,surface_pressure&start_date=2015-01-01&end_date=2025-12-31#hourly_weather_variables

* Cherrapunji (Sohra), India: (26.6613, 83.2679) https://open-meteo.com/en/docs/historical-weather-api?timezone=auto&latitude=26.6613&longitude=83.2679&hourly=temperature_2m,relative_humidity_2m,rain,wind_speed_10m,cloud_cover,surface_pressure&start_date=2015-01-01&end_date=2025-12-31#hourly_weather_variables

* Tutunendo, Colombia: (5.7432, -76.5409) https://open-meteo.com/en/docs/historical-weather-api?timezone=auto&latitude=5.7432&longitude=-76.5409&hourly=temperature_2m,relative_humidity_2m,rain,wind_speed_10m,cloud_cover,surface_pressure&start_date=2015-01-01&end_date=2025-12-31#hourly_weather_variables

* San Antonio de Ureca, Equatorial Guinea: (3.2485, 8.5669) https://open-meteo.com/en/docs/historical-weather-api?timezone=auto&latitude=3.2485&longitude=8.5669&hourly=temperature_2m,relative_humidity_2m,rain,wind_speed_10m,cloud_cover,surface_pressure&start_date=2015-01-01&end_date=2025-12-31#hourly_weather_variables

* Seoul, South Korea: https://open-meteo.com/en/docs/historical-weather-api?timezone=auto&latitude=37.566&longitude=126.9784&hourly=temperature_2m,relative_humidity_2m,rain,wind_speed_10m,cloud_cover,surface_pressure&start_date=2015-01-01&end_date=2025-12-31#hourly_weather_variables

* Kuala Lumput, Malaysia: https://open-meteo.com/en/docs/historical-weather-api?timezone=auto&latitude=3.1412&longitude=101.6865&hourly=temperature_2m,relative_humidity_2m,rain,wind_speed_10m,cloud_cover,surface_pressure&start_date=2015-01-01&end_date=2025-12-31#hourly_weather_variables

* Hanoi, Vietnam: https://open-meteo.com/en/docs/historical-weather-api?timezone=auto&latitude=21.0245&longitude=105.8412&hourly=temperature_2m,relative_humidity_2m,rain,wind_speed_10m,cloud_cover,surface_pressure&start_date=2015-01-01&end_date=2025-12-31#hourly_weather_variables


### Data collection

How to run:
* copy the code from open-meteo and run this section
* rename the generated csv data to data'n'
* repeat this section for all location

In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 21.0245,
	"longitude": 105.8412,
	"start_date": "2015-01-01",
	"end_date": "2025-12-31",
	"hourly": ["temperature_2m", "relative_humidity_2m", "rain", "wind_speed_10m", "cloud_cover", "surface_pressure"],
	"timezone": "auto",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_rain = hourly.Variables(2).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(3).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(4).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(5).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["rain"] = hourly_rain
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["surface_pressure"] = hourly_surface_pressure

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 21.054479598999023°N 105.8071060180664°E
Elevation: 10.0 m asl
Timezone: b'Asia/Bangkok'b'GMT+7'
Timezone difference to GMT+0: 25200s

Hourly data
                            date  temperature_2m  relative_humidity_2m  rain  \
0     2015-01-01 00:00:00+07:00       13.056500             84.516525   0.0   
1     2015-01-01 01:00:00+07:00       12.456500             87.318375   0.0   
2     2015-01-01 02:00:00+07:00       11.856501             90.530853   0.0   
3     2015-01-01 03:00:00+07:00       11.206500             92.012756   0.0   
4     2015-01-01 04:00:00+07:00       10.756500             93.228447   0.0   
...                         ...             ...                   ...   ...   
96427 2025-12-31 19:00:00+07:00       20.900000             81.169083   0.0   
96428 2025-12-31 20:00:00+07:00       20.650000             83.473373   0.0   
96429 2025-12-31 21:00:00+07:00       20.100000             87.450111   0.0   
96430 2025-12-31 22:00:00+07:00       19.700001  

In [ ]:
daily_dataframe = pd.DataFrame()

daily_dataframe['date'] = hourly_dataframe['date'].dt.date.unique()

# Calculate daily aggregates
daily_temp_agg = hourly_dataframe.groupby(hourly_dataframe['date'].dt.date)['temperature_2m'].agg(
    MaxTemp='max',
    MinTemp='min',
    MeanTemp='mean'
).reset_index()
daily_temp_agg.rename(columns={'date': 'date'}, inplace=True)

daily_rainfall = hourly_dataframe.groupby(hourly_dataframe['date'].dt.date)['rain'].sum().reset_index()
daily_rainfall.rename(columns={'rain': 'TotalRainfall', 'date': 'date'}, inplace=True)

daily_dataframe = pd.merge(daily_dataframe, daily_temp_agg, on='date', how='left')
daily_dataframe = pd.merge(daily_dataframe, daily_rainfall, on='date', how='left')


# Function to get hourly data for a specific column and hour
def get_hourly_value(df, column_name, hour):
    filtered_df = df[df['date'].dt.hour == hour][['date', column_name]].copy()
    filtered_df['date'] = filtered_df['date'].dt.date
    filtered_df.rename(columns={column_name: f'{column_name.replace('_', '').replace('2m','').replace('10m','').replace('cover','').capitalize()}{hour:02}am' if hour < 12 else f'{column_name.replace('_', '').replace('2m','').replace('10m','').replace('cover','').capitalize()}{hour-12:02}pm'},
                 inplace=True)
    return filtered_df


# Extract specific hourly values
hours = [0, 3, 6, 9, 12, 15, 18, 21] # 12am, 3am, 6am, 9am, 12pm, 3pm, 6pm, 9pm

for hour in hours:
    # Humidity
    hourly_humidity = get_hourly_value(hourly_dataframe, 'relative_humidity_2m', hour)
    daily_dataframe = pd.merge(daily_dataframe, hourly_humidity, on='date', how='left')

    # Pressure
    hourly_pressure = get_hourly_value(hourly_dataframe, 'surface_pressure', hour)
    daily_dataframe = pd.merge(daily_dataframe, hourly_pressure, on='date', how='left')

    # Wind Speed
    hourly_windspeed = get_hourly_value(hourly_dataframe, 'wind_speed_10m', hour)
    daily_dataframe = pd.merge(daily_dataframe, hourly_windspeed, on='date', how='left')

    # Cloud Cover
    hourly_cloud = get_hourly_value(hourly_dataframe, 'cloud_cover', hour)
    daily_dataframe = pd.merge(daily_dataframe, hourly_cloud, on='date', how='left')

In [ ]:
print("\nDaily data\n", daily_dataframe)



Daily data
             date    MaxTemp    MinTemp   MeanTemp  TotalRainfall  \
0     2015-01-01  22.806499   9.656500  16.129417            0.0   
1     2015-01-02  21.406500  10.706500  15.764834            0.0   
2     2015-01-03  19.856501  11.156500  16.027334            0.0   
3     2015-01-04  22.256500  16.406500  18.733583            0.2   
4     2015-01-05  25.106501  17.756500  20.756500            0.0   
...          ...        ...        ...        ...            ...   
4013  2025-12-27  19.450001  12.550000  15.795834            0.0   
4014  2025-12-28  20.950001  13.850000  17.343750            0.0   
4015  2025-12-29  23.299999  16.799999  19.579166            0.2   
4016  2025-12-30  23.799999  17.600000  20.408333            0.2   
4017  2025-12-31  24.700001  17.799999  20.768751            0.4   

      Relativehumidity00am  Surfacepressure00am  Windspeed00am  Cloud00am  \
0                84.516525          1021.580078       7.895416        0.0   
1               

In [ ]:
# Calculate HeavyRainTomorrow
daily_dataframe['HeavyRainTomorrow'] = (daily_dataframe['TotalRainfall'].shift(-1) > 50).astype(int)

In [ ]:
print("\nDaily data with HeavyRainTomorrow\n", daily_dataframe)


Daily data with HeavyRainTomorrow
             date    MaxTemp    MinTemp   MeanTemp  TotalRainfall  \
0     2015-01-01  22.806499   9.656500  16.129417            0.0   
1     2015-01-02  21.406500  10.706500  15.764834            0.0   
2     2015-01-03  19.856501  11.156500  16.027334            0.0   
3     2015-01-04  22.256500  16.406500  18.733583            0.2   
4     2015-01-05  25.106501  17.756500  20.756500            0.0   
...          ...        ...        ...        ...            ...   
4013  2025-12-27  19.450001  12.550000  15.795834            0.0   
4014  2025-12-28  20.950001  13.850000  17.343750            0.0   
4015  2025-12-29  23.299999  16.799999  19.579166            0.2   
4016  2025-12-30  23.799999  17.600000  20.408333            0.2   
4017  2025-12-31  24.700001  17.799999  20.768751            0.4   

      Relativehumidity00am  Surfacepressure00am  Windspeed00am  Cloud00am  \
0                84.516525          1021.580078       7.895416        

In [ ]:
total_rainy_days = (daily_dataframe['TotalRainfall'] > 0).sum()
percentage_rainy_days = (total_rainy_days / len(daily_dataframe)) * 100
print(f"Percentage of total rainy days: {percentage_rainy_days:.2f}%")

Percentage of total rainy days: 68.67%


In [ ]:
percentage_heavy_rain = (daily_dataframe['HeavyRainTomorrow'].sum() / len(daily_dataframe)) * 100
print(f"Percentage of heavy rain days: {percentage_heavy_rain:.2f}%")

Percentage of heavy rain days: 1.24%


In [ ]:
# Calculate the 90th percentile of TotalRainfall
rainfall_90th_percentile = daily_dataframe['TotalRainfall'].quantile(0.90)

# Create Rain10Tomorrow column
daily_dataframe['Rain10Tomorrow'] = (daily_dataframe['TotalRainfall'].shift(-1) >= rainfall_90th_percentile).astype(int)

In [ ]:
print(f"\nLowest rainfall value for top 10% (90th percentile): {rainfall_90th_percentile:.2f}mm")
print("\nDaily data with Rain10Tomorrow\n", daily_dataframe)


Lowest rainfall value for top 10% (90th percentile): 17.83mm

Daily data with Rain10Tomorrow
             date    MaxTemp    MinTemp   MeanTemp  TotalRainfall  \
0     2015-01-01  22.806499   9.656500  16.129417            0.0   
1     2015-01-02  21.406500  10.706500  15.764834            0.0   
2     2015-01-03  19.856501  11.156500  16.027334            0.0   
3     2015-01-04  22.256500  16.406500  18.733583            0.2   
4     2015-01-05  25.106501  17.756500  20.756500            0.0   
...          ...        ...        ...        ...            ...   
4013  2025-12-27  19.450001  12.550000  15.795834            0.0   
4014  2025-12-28  20.950001  13.850000  17.343750            0.0   
4015  2025-12-29  23.299999  16.799999  19.579166            0.2   
4016  2025-12-30  23.799999  17.600000  20.408333            0.2   
4017  2025-12-31  24.700001  17.799999  20.768751            0.4   

      Relativehumidity00am  Surfacepressure00am  Windspeed00am  Cloud00am  \
0          

In [ ]:
daily_dataframe.to_csv('daily_weather_data.csv', index=False)
print("daily_weather_data.csv saved successfully!")

daily_weather_data.csv saved successfully!


In [ ]:
print(f"Percentage of total rainy days: {percentage_rainy_days:.2f}%")
print(f"Percentage of heavy rain days: {percentage_heavy_rain:.2f}%")
print(f"\nLowest rainfall value for top 10% (90th percentile): {rainfall_90th_percentile:.2f}mm")

Percentage of total rainy days: 68.67%
Percentage of heavy rain days: 1.24%

Lowest rainfall value for top 10% (90th percentile): 17.83mm


### Combining all csv files (Run this section last)

In [ ]:
import glob

# Get a list of all CSV files in the current directory
csv_files = glob.glob('*.csv')

# Create an empty list to store dataframes
df_list = []

# Loop through the list of csv files and read each into a dataframe
for file in csv_files:
    if file == 'daily_weather_data.csv': # Skip the output file from the previous step if it exists
        continue
    df = pd.read_csv(file)
    df_list.append(df)

# Concatenate all dataframes into one
combined_df = pd.concat(df_list, ignore_index=True)

# Display the first 5 rows of the combined dataframe
print("Combined DataFrame:")
display(combined_df)

Combined DataFrame:


,date,MaxTemp,MinTemp,MeanTemp,TotalRainfall,Relativehumidity00am,Surfacepressure00am,Windspeed00am,Cloud00am,Relativehumidity03am,...,Relativehumidity06pm,Surfacepressure06pm,Windspeed06pm,Cloud06pm,Relativehumidity09pm,Surfacepressure09pm,Windspeed09pm,Cloud09pm,HeavyRainTomorrow,Rain10Tomorrow
0,2015-01-01,-4.036,-8.486000,-6.692250,0.0,35.259354,1018.76770,21.407139,0.0,32.687237,...,38.985610,1021.46893,14.512064,2.0,48.663970,1022.84850,12.904882,5.0,0,0
1,2015-01-02,-1.686,-7.486000,-5.196417,0.0,50.996235,1022.55200,11.720751,94.0,51.067000,...,32.183067,1022.58250,10.009036,0.0,38.726048,1023.85680,8.714677,0.0,0,0
2,2015-01-03,1.864,-8.535999,-3.625583,0.0,45.030655,1023.84015,6.193674,0.0,44.715717,...,72.227960,1018.03810,7.993298,83.0,75.274864,1017.13860,5.600286,100.0,0,0
3,2015-01-04,7.364,-2.136000,1.684833,0.0,79.950970,1016.14014,6.989936,72.0,80.911354,...,90.202210,1014.90690,10.028439,22.0,97.528824,1016.28730,4.394360,7.0,0,0
4,2015-01-05,8.314,-2.336000,2.257750,1.0,99.266430,1016.04175,5.804825,100.0,98.177280,...,61.907803,1011.45245,5.191994,100.0,77.188675,1009.32180,1.138420,100.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28121,2025-12-27,19.450,12.550000,15.795834,0.0,82.358444,1018.88560,5.474486,14.0,86.225330,...,74.948710,1017.19885,3.640275,58.0,80.216050,1018.29500,3.947708,4.0,0,0
28122,2025-12-28,20.950,13.850000,17.343750,0.0,86.091250,1018.39030,1.094897,27.0,84.615770,...,72.020230,1015.51135,1.548418,5.0,79.745740,1016.60650,3.710795,70.0,0,0
28123,2025-12-29,23.300,16.800000,19.579166,0.2,81.956955,1016.00240,7.353611,91.0,86.617890,...,75.866750,1012.72090,14.182355,100.0,82.268650,1014.71450,10.883676,100.0,0,0
28124,2025-12-30,23.800,17.600000,20.408333,0.2,87.342050,1014.71250,5.863139,100.0,90.994870,...,72.506680,1011.12720,13.337016,88.0,85.238080,1012.41864,12.031756,100.0,0,0


In [ ]:
# Ensure 'date' column is in datetime format
combined_df['date'] = pd.to_datetime(combined_df['date'])

# Calculate total rainy day percentage
total_rainy_days_combined = (combined_df['TotalRainfall'] > 0).sum()
percentage_rainy_days_combined = (total_rainy_days_combined / len(combined_df)) * 100
print(f"Combined Data - Percentage of total rainy days: {percentage_rainy_days_combined:.2f}%")

# Calculate HeavyRainTomorrow for combined_df
combined_df['HeavyRainTomorrow'] = (combined_df['TotalRainfall'].shift(-1) > 50).astype(int)
percentage_heavy_rain_combined = (combined_df['HeavyRainTomorrow'].sum() / len(combined_df)) * 100
print(f"Combined Data - Percentage of heavy rain days: {percentage_heavy_rain_combined:.2f}%")

# Calculate the 90th percentile of TotalRainfall for combined_df
rainfall_90th_percentile_combined = combined_df['TotalRainfall'].quantile(0.90)
print(f"Combined Data - Lowest rainfall value for top 10% (90th percentile): {rainfall_90th_percentile_combined:.2f}mm")

Combined Data - Percentage of total rainy days: 73.58%
Combined Data - Percentage of heavy rain days: 4.54%
Combined Data - Lowest rainfall value for top 10% (90th percentile): 31.40mm


In [ ]:
combined_df.to_csv('combined_weather_data.csv', index=False)
print("combined_weather_data.csv saved successfully!")

combined_weather_data.csv saved successfully!
